## Scaling

In [1]:
import jax, jax.numpy as jnp
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme('paper')

from untangle import algorithm 
from untangle.utils import collect_information, outputs_error
from untangle.scaler import *

key = jax.random.key(0)

### Similar Outputs

In [2]:
def f(x):
    x1, x2 = x
    return jnp.array([
         jnp.cos(4 * jnp.pi * x1 + x2) + x1**2,
        -jnp.sin(3 * jnp.pi * x2) + 3*x2**2,
    ])

m = 2
n = 2
rank = 2

In [3]:
num_points = 30
X, Y, J = collect_information(f, num_points, m, key=key)

In [4]:
learned_f = algorithm.cmtf_bsd(X, Y, J, rank, key=key)

Computing CMTF-BSD: 100%|█████████████████████████████████████████████████| 50/50 [00:04<00:00, 11.78it/s]


In [5]:
errors = outputs_error(f, learned_f, X)
print(f'error(y1) = {errors[0]:.1f}%, error(y2) = {errors[1]:.1f}%')

error(y1) = 3.1%, error(y2) = 2.1%


### Different Magnitudes

In [6]:
def f2(x):
    x1, x2 = x
    return jnp.array([
         100*(jnp.cos(4 * jnp.pi * x1 + x2) + x1**2),
        -jnp.sin(3 * jnp.pi * x2) + 3*x2**2,
    ])

In [7]:
num_points = 30
X, Y, J = collect_information(f2, num_points, m, key=key)

In [8]:
learned_f = algorithm.cmtf_bsd(X, Y, J, rank, key=key)

Computing CMTF-BSD: 100%|█████████████████████████████████████████████████| 50/50 [00:02<00:00, 17.22it/s]


In [9]:
errors = outputs_error(f2, learned_f, X)
print(f'error(y1) = {errors[0]:.0f}%, error(y2) = {errors[1]:.0f}%')

error(y1) = 2%, error(y2) = 100%


### Tensor Scaling

In [10]:
factors, J_scaled, Y_scaled = scale_tensor(J, Y)
jnp.linalg.norm(J_scaled[0, :, :]), jnp.linalg.norm(J_scaled[1, :, :])

(Array(7.7459674, dtype=float32), Array(7.745967, dtype=float32))

In [11]:
learned_f_scaled = algorithm.cmtf_bsd(X, Y_scaled, J_scaled, rank, key=key)
learned_f = lambda x: learned_f_scaled(x) / factors

Computing CMTF-BSD: 100%|█████████████████████████████████████████████████| 50/50 [00:03<00:00, 16.47it/s]


In [12]:
errors = outputs_error(f2, learned_f, X)
print(f'error(y1) = {errors[0]:.1f}%, error(y2) = {errors[1]:.1f}%')

error(y1) = 3.1%, error(y2) = 2.1%
